# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [8]:
col = "Outstanding_Debt"

## <font color = 'skyblue'> ANÁLISIS GENERAL

Las medianas tienen el orden esperado: Median Bad > Mediana Standard > Mediana Good.

Esto concuerda con la hipótesis de que a mayor endeudamiento mayor cantidad de malos deudores.

In [9]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [10]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [11]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Outstanding_Debt_Decile,,,,,,,,,,
0,0.23,230.44,10000,0.1,0,4872,5128,0.0000,0.4872,0.5128
1,230.77,457.92,10000,0.1,0,4576,5424,0.0000,0.4576,0.5424
2,458.36,688.84,10000,0.1,0,4784,5216,0.0000,0.4784,0.5216
3,688.99,923.85,10000,0.1,0,4808,5192,0.0000,0.4808,0.5192
4,924.10,1166.08,10000,0.1,0,4600,5400,0.0000,0.4600,0.5400
5,1166.23,1354.33,10000,0.1,776,3944,5280,0.0776,0.3944,0.5280
6,1354.35,1624.60,10000,0.1,1880,2800,5320,0.1880,0.2800,0.5320
7,1625.40,2297.22,10000,0.1,4424,0,5576,0.4424,0.0000,0.5576
8,2297.52,3189.00,10000,0.1,6688,0,3312,0.6688,0.0000,0.3312


No missing values found.
No infinite values found.
No duplicate rows found.


In [13]:
df[(df[continuous_variable] >= 1354.35) & (df[continuous_variable] <= 1624.60) & (df['Credit_Score'] == 1)].shape

(5320, 85)

Proporción de Buenos: se observa una relación negativa entre el saldo del crédito remanente y la proporción de buenos deudores, lo cual es razonable.

Proporción de standard: esta relación no es tan clara hasta saldos cercanos a los USD 2.300, pero luego de este umbral, sí, lo cual es razonable.

Proporción de malos: la proporción de malos y el saldo remanente del crédito tienen la relación positiva esperada.

In [15]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y la variable:

In [16]:
group_map = {0: "Group_1", 
             1: "Group_1", 
             2: "Group_1", 
             3: "Group_1", 
             4: "Group_1",
             5: "Group_1", 
             6: "Group_2",
             7: "Group_3",
             8: "Group_4",
             9: "Group_5"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3,
    'Group_4': 4,
    'Group_5': 5
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Outstanding_Debt,,,,,,,,,,
Group_1,0.23,1354.33,60000,0.6,776,27584,31640,0.012933,0.459733,0.527333
Group_2,1354.35,1624.60,10000,0.1,1880,2800,5320,0.188000,0.280000,0.532000
Group_3,1625.40,2297.22,10000,0.1,4424,0,5576,0.442400,0.000000,0.557600
Group_4,2297.52,3189.00,10000,0.1,6688,0,3312,0.668800,0.000000,0.331200
Group_5,3189.70,4998.07,10000,0.1,10000,0,0,1.000000,0.000000,0.000000


No missing values found.
No infinite values found.
No duplicate rows found.


In [17]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [18]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [19]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Outstanding_Debt,"100,000.00","1,426.22","1,155.13",0.23,566.07,"1,166.15","1,945.96","4,998.07"


Todos los coeficientes son significativos.

Outstanding_Debt_Scaled -9.5728: según lo esperado, el coeficiente es negativo: por cada dólara adicional de deuda, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -4.4553: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 1.2099: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [20]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.736532
         Iterations: 14
         Function evaluations: 16
         Gradient evaluations: 16
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -73653.
Model:                   OrderedModel   AIC:                         1.473e+05
Method:            Maximum Likelihood   BIC:                         1.473e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        12:47:40                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                              coef    std

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Outstanding_Debt_Decile -0.5579: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -4.2157: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 1.1201: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [22]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.812843
         Iterations: 12
         Function evaluations: 14
         Gradient evaluations: 14
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -81284.
Model:                   OrderedModel   AIC:                         1.626e+05
Method:            Maximum Likelihood   BIC:                         1.626e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        12:50:59                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                              coef    std

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Outstanding_Debt -1.6317: Por cada grupo adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -5.3177: Umbral que separa las categorías Bad y Standard.

Threshold 1/2 1.3351: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [23]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.708435
         Iterations: 13
         Function evaluations: 14
         Gradient evaluations: 14
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -70844.
Model:                   OrderedModel   AIC:                         1.417e+05
Method:            Maximum Likelihood   BIC:                         1.417e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        12:51:09                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                               coef    st

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Outstanding_Debt` usando Regresión Ordinal

| Representación                       | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|-------------------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Outstanding_Debt_Scaled`           | -9.5728               | **Log-Likelihood**: -73,653<br>**AIC**: 147,306<br>**BIC**: 147,347                    | 🔹 Ajuste intermedio.<br>🔹 Alta relación negativa con el score.<br>🔹 Captura bien la variabilidad continua. |
| `Outstanding_Debt_Decile`           | -0.5579               | **Log-Likelihood**: -81,284<br>**AIC**: 162,568<br>**BIC**: 162,609                    | 🔹 Peor ajuste.<br>🔹 Discretización reduce precisión.<br>🔹 Útil si se desea mayor interpretabilidad. |
| `Grouped_Outstanding_Debt`          | -1.6317               | **Log-Likelihood**: -70,844<br>**AIC**: 141,688<br>**BIC**: 141,729                    | 🔹 Mejor ajuste.<br>🔹 Buena relación negativa y mejor balance entre simplicidad y desempeño. |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [42]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
